# Lab 17-12 · Sharing the Spectrum

Six parts, each built on `access.py`. **Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import heapq
import math
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import access as ac
rng = np.random.default_rng(1712)

## Part 1 · Reuse

*Predict:* the smallest cluster size giving 12 dB edge SIR for path-loss exponents 3, 3.5 and 4.

In [2]:
for n in (3, 3.5, 4):
    row = [(N, round(10 * math.log10(ac.sir_first_tier(N, n)), 1)) for N in ac.valid_cluster_sizes(12)]
    best = min(N for N, s in row if s >= 12)
    print(f"n = {n}: {row}  -> smallest cluster for 12 dB: {best}")

n = 3: [(1, -0.6), (3, 6.5), (4, 8.4), (7, 12.1), (9, 13.7), (12, 15.6)]  -> smallest cluster for 12 dB: 7
n = 3.5: [(1, 0.6), (3, 8.9), (4, 11.1), (7, 15.4), (9, 17.3), (12, 19.5)]  -> smallest cluster for 12 dB: 7
n = 4: [(1, 1.8), (3, 11.3), (4, 13.8), (7, 18.7), (9, 20.8), (12, 23.3)]  -> smallest cluster for 12 dB: 4


## Part 2 · ALOHA

*Predict:* where each ALOHA variant peaks, and its peak throughput.

In [3]:
G = np.linspace(0.05, 3, 60)
plt.plot(G, [ac.aloha(g) for g in G], label="pure (analytic)")
plt.plot(G, [ac.slotted_aloha(g) for g in G], label="slotted (analytic)")
gs = np.array([0.25, 0.5, 1, 1.5, 2, 3])
plt.plot(gs, [ac.simulate_pure_aloha(g, 100_000, rng) for g in gs], "o", label="pure (simulated)")
plt.plot(gs, [ac.simulate_slotted_aloha(g, 100_000, rng) for g in gs], "s", label="slotted (simulated)")
plt.xlabel("G"); plt.ylabel("S"); plt.legend(); plt.show()
print(f"peaks: pure {ac.aloha(0.5):.3f} at G = 0.5, slotted {ac.slotted_aloha(1):.3f} at G = 1")

peaks: pure 0.184 at G = 0.5, slotted 0.368 at G = 1


## Part 3 · CSMA/CA windows

*Predict:* which of $W$ = 16 and $W$ = 64 serves 5 stations better, and which serves 50 better.

In [4]:
for n in (5, 50):
    for w in (16, 64):
        tau = ac.tau_from_window(w)
        sim = ac.simulate_slots(n, tau, 200_000, rng)
        print(f"n = {n:2d}, W = {w}: idle/success/collision simulated {np.round(sim, 3)}, formula {np.round(ac.slot_outcomes(n, tau), 3)}")

n =  5, W = 16: idle/success/collision simulated [0.534 0.358 0.108], formula [0.535 0.357 0.109]
n =  5, W = 64: idle/success/collision simulated [0.855 0.136 0.009], formula [0.855 0.136 0.009]
n = 50, W = 16: idle/success/collision simulated [0.002 0.013 0.985], formula [0.002 0.013 0.985]
n = 50, W = 64: idle/success/collision simulated [0.21  0.333 0.457], formula [0.21  0.333 0.458]


## Part 4 · Erlang B

*Predict:* the blocking of 10 channels at 5 and 7 Erlangs; simulate a loss system.

In [5]:
def loss_system(c, a, ncalls):
    t, busy, blocked = 0.0, [], 0
    for _ in range(ncalls):
        t += rng.exponential(1 / a)
        while busy and busy[0] <= t:
            heapq.heappop(busy)
        if len(busy) >= c:
            blocked += 1
        else:
            heapq.heappush(busy, t + rng.exponential(1.0))
    return blocked / ncalls
for a in (5, 7):
    print(f"A = {a} Erl: simulated {100 * loss_system(10, a, 200_000):.2f}%, Erlang B {100 * ac.erlang_b(10, a):.2f}%")

A = 5 Erl: simulated 1.93%, Erlang B 1.84%


A = 7 Erl: simulated 7.73%, Erlang B 7.87%


## Part 5 · Schedulers

*Predict:* which scheduler gives the highest sum rate, and which the most even split, for users at 20, 15, 10, 5 and 0 dB.

In [6]:
res = ac.simulate_schedulers([20, 15, 10, 5, 0], 40_000, rng)
for k, name in (("rr", "round robin"), ("max", "max-SNR"), ("pf", "proportional fair")):
    print(f"{name:17s}: per user {np.round(res[k], 3)}, sum {res[k].sum():.2f}, Jain {ac.jain(res[k]):.2f}")

round robin      : per user [1.172 0.872 0.582 0.343 0.171], sum 3.14, Jain 0.75
max-SNR          : per user [4.823 1.262 0.174 0.009 0.   ], sum 6.27, Jain 0.32
proportional fair: per user [1.6   1.235 0.88  0.564 0.313], sum 4.59, Jain 0.80


## Part 6 · HARQ

*Predict:* throughput and residual error for $p$ = 0.1 and 0.3 with up to 1, 2 and 4 transmissions.

In [7]:
for p in (0.1, 0.3):
    for m in (1, 2, 4):
        thr, res_err = ac.harq_throughput(p, m)
        tx = ok = 0
        for _ in range(100_000):
            for _k in range(m):
                tx += 1
                if rng.random() >= p:
                    ok += 1
                    break
        print(f"p = {p}, M = {m}: throughput formula {thr:.3f}, simulated {ok / tx:.3f}; residual {res_err:.4f}, simulated {1 - ok / 100_000:.4f}")

p = 0.1, M = 1: throughput formula 0.900, simulated 0.899; residual 0.1000, simulated 0.1010
p = 0.1, M = 2: throughput formula 0.900, simulated 0.900; residual 0.0100, simulated 0.0101
p = 0.1, M = 4: throughput formula 0.900, simulated 0.900; residual 0.0001, simulated 0.0001


p = 0.3, M = 1: throughput formula 0.700, simulated 0.697; residual 0.3000, simulated 0.3031
p = 0.3, M = 2: throughput formula 0.700, simulated 0.701; residual 0.0900, simulated 0.0888


p = 0.3, M = 4: throughput formula 0.700, simulated 0.700; residual 0.0081, simulated 0.0079
